In [2]:
import ROOT as R
import pandas as pd

data_files = "~/hps/track_cluster_matching/data/*.root"
mc_files = "~/hps/ecal_calibration/data/fee_mc/*.root"

df_data = R.RDataFrame("MiniDST", data_files)
df_mc = R.RDataFrame("MiniDST", mc_files)

print("Events in data:", df_data.Count().GetValue())
print("Events in MC:", df_mc.Count().GetValue())

Events in data: 1918784
Events in MC: 617450


In [ ]:
# Trigger check
print("=== Dataset Summary ===")
print(f"Total events      : {df_data.Count().GetValue():,}")

runs = df_data.AsNumpy(["run_number"])["run_number"]
print(f"Unique runs       : {sorted(set(runs))}")
print(f"Run range         : {runs.min()} — {runs.max()}")

for name, bit in [("Pair_0 (A-prime)", 8), ("Pair_1 (Moller)", 9), 
                  ("FEE Top", 18), ("FEE Bot", 19), ("Pulser", 15)]:
    n = df_data.Filter(f"trigger & (1u << {bit})").Count().GetValue()
    print(f"{name:<20}: {n:,}")

=== Dataset Summary ===
Total events      : 1,918,784
Unique runs       : [np.int32(14185), np.int32(14193)]
Run range         : 14185 — 14193
Pair_0 (A-prime)    : 7,538
Pair_1 (Moller)     : 0
FEE Top             : 2,006
FEE Bot             : 2,059
Pulser              : 183


In [5]:
# Get all column names as a list of strings
all_columns = [str(c) for c in df_data.GetColumnNames()]
ecal_cols = [c for c in all_columns if "ecal" in c]
print(f"Found {len(ecal_cols)} columns:")
print(ecal_cols)

Found 26 columns:
['ecal_cluster_energy', 'ecal_cluster_hits', 'ecal_cluster_nhits', 'ecal_cluster_seed_energy', 'ecal_cluster_seed_index', 'ecal_cluster_seed_ix', 'ecal_cluster_seed_iy', 'ecal_cluster_time', 'ecal_cluster_x', 'ecal_cluster_y', 'ecal_cluster_z', 'ecal_hit_energy', 'ecal_hit_index_x', 'ecal_hit_index_y', 'ecal_hit_time', 'ecal_hit_x', 'ecal_hit_y', 'ecal_hit_z', 'part_ecal_cluster', 'track_x_at_ecal', 'track_y_at_ecal', 'track_z_at_ecal', 'v0_em_pos_ecal_x', 'v0_em_pos_ecal_y', 'v0_ep_pos_ecal_x', 'v0_ep_pos_ecal_y']


In [9]:
cols = ["part_ecal_cluster", "ecal_cluster_energy", "ecal_cluster_seed_energy",
        "ecal_cluster_seed_index", "ecal_hit_index_x", "ecal_cluster_seed_ix"]

tmp = df_mc.Range(10).AsNumpy(cols)

for i in range(10):
    print(f"\n--- Event {i} ---")
    for col in cols:
        print(f"  {col}: len={len(tmp[col][i])}  values={list(tmp[col][i])}")


--- Event 0 ---
  part_ecal_cluster: len=2  values=[0, 0]
  ecal_cluster_energy: len=1  values=[3.828559398651123]
  ecal_cluster_seed_energy: len=1  values=[2.1737289428710938]
  ecal_cluster_seed_index: len=1  values=[7]
  ecal_hit_index_x: len=11  values=[3, -1, -1, 1, 2, -1, 1, 1, 3, 2, 2]
  ecal_cluster_seed_ix: len=1  values=[1]

--- Event 1 ---
  part_ecal_cluster: len=2  values=[0, 0]
  ecal_cluster_energy: len=1  values=[3.9783241748809814]
  ecal_cluster_seed_energy: len=1  values=[2.0931081771850586]
  ecal_cluster_seed_index: len=1  values=[1]
  ecal_hit_index_x: len=11  values=[20, 19, 20, 17, 19, 20, 17, 19, 18, 18, 18]
  ecal_cluster_seed_ix: len=1  values=[19]

--- Event 2 ---
  part_ecal_cluster: len=0  values=[]
  ecal_cluster_energy: len=0  values=[]
  ecal_cluster_seed_energy: len=0  values=[]
  ecal_cluster_seed_index: len=0  values=[]
  ecal_hit_index_x: len=1  values=[18]
  ecal_cluster_seed_ix: len=0  values=[]

--- Event 3 ---
  part_ecal_cluster: len=2  value

In [25]:
# Look at event 5 (0-indexed)
N = 0

cols = ["part_pdg","part_track", "part_ecal_cluster", "ecal_cluster_energy", "ecal_cluster_seed_ix", "track_px"]

display = df_mc.Range(N, N+1).Display(cols)
display.Print()

+-----+----------+------------+-------------------+---------------------+----------------------+-----+
| Row | part_pdg | part_track | part_ecal_cluster | ecal_cluster_energy | ecal_cluster_seed_ix | ... | 
+-----+----------+------------+-------------------+---------------------+----------------------+-----+
| 0   | 22       | -1         | 0                 | 3.828559            | 1                    | ... | 
|     | 22       | -1         | 0                 |                     |                      | ... | 
+-----+----------+------------+-------------------+---------------------+----------------------+-----+


Info in <Print>: Only showing 6 columns out of 7



In [4]:
df = df_data

display = df.Display(["ecal_cluster_seed_ix", "part_ecal_cluster", "ecal_cluster_seed_index", "part_pdg"], 10) # 10 is the number of rows
display.Print()


# print("minimum seed ix:", df.Min("ecal_cluster_seed_ix").GetValue())
# print("maximum seed ix:", df.Max("ecal_cluster_seed_ix").GetValue())
# print("minimum seed iy:", df.Min("ecal_cluster_seed_iy").GetValue())
# print("maximum seed iy:", df.Max("ecal_cluster_seed_iy").GetValue())

+-----+----------------------+-------------------+-------------------------+----------+
| Row | ecal_cluster_seed_ix | part_ecal_cluster | ecal_cluster_seed_index | part_pdg | 
+-----+----------------------+-------------------+-------------------------+----------+
| 0   | -15                  | 1                 | 3                       | -11      | 
|     | 23                   | -1                | 8                       | 11       | 
|     |                      | 0                 |                         | 11       | 
+-----+----------------------+-------------------+-------------------------+----------+
| 1   | -14                  | -1                | 22                      | 11       | 
|     | 13                   | -1                | 15                      | 11       | 
|     | -14                  | 2                 | 5                       | 11       | 
|     | -2                   | -1                | 1                       | -11      | 
|     |                 

In [5]:
# Histo1D((name, title; x-axis; y-axis, nbins, x_low, x_high), "column_name")

col_name = "ecal_cluster_seed_index"
c = R.TCanvas("c", f"{col_name}", 800, 600)
h = df_mc.Histo1D(("h_{col_name}", f"{col_name}; {col_name}; Counts", 16, -2, 6), col_name)
# h.Draw()
# c.Draw()

unique_entries = []
for i in range(1, h.GetNbinsX() + 1):
    if h.GetBinContent(i) > 0:
        unique_entries.append(h.GetBinCenter(i))

print(unique_entries)

[0.25, 1.25, 2.25, 3.25, 4.25, 5.25]


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c


In [ ]:
# ----------------------------
# Helper: build FEE-like vectors (Eclus, p, E/p, ix, iy)
# ----------------------------
R.gInterpreter.Declare(r"""
#include <ROOT/RVec.hxx>
#include <cmath>
using ROOT::VecOps::RVec;

struct FeeVars {
  RVec<float> Eclus;
  RVec<float> p;
  RVec<float> EoverP;
  RVec<int>   ix;
  RVec<int>   iy;
};

// Build FEE-like candidates using part -> (track,cluster) indices.
FeeVars make_fee_vars(
    const RVec<int>&   part_pdg,
    const RVec<int>&   part_track,
    const RVec<int>&   part_ecal_cluster,
    const RVec<float>& track_px,
    const RVec<float>& track_py,
    const RVec<float>& track_pz,
    const RVec<float>& ecal_cluster_energy,
    const RVec<int>&   ecal_cluster_seed_ix,
    const RVec<int>&   ecal_cluster_seed_iy,
    float pmin,
    float Emin,
    float eop_min,
    float eop_max
){
  FeeVars out;
  const int ntr = (int)track_px.size();
  const int ncl = (int)ecal_cluster_energy.size();

  for (size_t i=0; i<part_pdg.size(); ++i){
    if (part_pdg[i] != 11) continue; // e-

    int tr = part_track[i];
    int cl = part_ecal_cluster[i];
    if (tr < 0 || cl < 0) continue;
    if (tr >= ntr || cl >= ncl) continue;

    const float px = track_px[tr];
    const float py = track_py[tr];
    const float pz = track_pz[tr];
    const float psum = std::sqrt(px*px + py*py + pz*pz);

    const float E = ecal_cluster_energy[cl];
    if (psum <= 0) continue;

    const float eop = E / psum;

    // loose FEE-like cuts
    if (psum < pmin) continue;
    if (E   < Emin) continue;
    if (eop < eop_min || eop > eop_max) continue;

    out.Eclus.push_back(E);
    out.p.push_back(psum);
    out.EoverP.push_back(eop);

    // seed ix/iy useful later for per-crystal calibration
    if ((int)ecal_cluster_seed_ix.size() > cl){
      out.ix.push_back(ecal_cluster_seed_ix[cl]);
      out.iy.push_back(ecal_cluster_seed_iy[cl]);
    } else {
      out.ix.push_back(999);
      out.iy.push_back(999);
    }
  }
  return out;
}
""")

# ----------------------------
# Choose initial loose cut values
# ----------------------------
PMIN = 3.3     # or 3.3
EMIN = 0.3
EOP_MIN = 0.7
EOP_MAX = 1.3

def add_fee_columns(df):
    return (df
        .Define("fee", f"make_fee_vars(part_pdg, part_track, part_ecal_cluster, "
                       f"track_px, track_py, track_pz, "
                       f"ecal_cluster_energy, ecal_cluster_seed_ix, ecal_cluster_seed_iy, "
                       f"{PMIN}f, {EMIN}f, {EOP_MIN}f, {EOP_MAX}f)")
        .Define("fee_Eclus", "fee.Eclus")
        .Define("fee_p",     "fee.p")
        .Define("fee_EoP",   "fee.EoverP")
        .Define("fee_ix",    "fee.ix")
        .Define("fee_iy",    "fee.iy")
    )

dfd = add_fee_columns(df_data)
dfm = add_fee_columns(df_mc)

# ----------------------------
# Histograms: cluster energy and E/p
# ----------------------------
hE_data = dfd.Histo1D(("hE_data", "FEE-like cluster energy (DATA); E_{clus} [GeV]; Counts", 200, 0, 4.5), "fee_Eclus")
hE_mc   = dfm.Histo1D(("hE_mc",   "FEE-like cluster energy (MC); E_{clus} [GeV]; Counts",   200, 0, 4.5), "fee_Eclus")

hEoP_data = dfd.Histo1D(("hEoP_data", "FEE-like E/p (DATA); E/p; Counts", 200, 0, 2.0), "fee_EoP")
hEoP_mc   = dfm.Histo1D(("hEoP_mc",   "FEE-like E/p (MC); E/p; Counts",   200, 0, 2.0), "fee_EoP")

# Trigger loops
hdE  = hE_data.GetPtr()
hmE  = hE_mc.GetPtr()
hdR  = hEoP_data.GetPtr()
hmR  = hEoP_mc.GetPtr()

# ----------------------------
# Draw: 2x2 comparison
# ----------------------------
c = R.TCanvas("c_fee", "FEE-like sanity plots", 1200, 900)
c.Divide(2,2)

c.cd(1); R.gPad.SetGrid(); hdE.SetLineWidth(2); hdE.Draw("HIST")
c.cd(2); R.gPad.SetGrid(); hmE.SetLineWidth(2); hmE.SetLineWidth(2); hmE.SetLineColor(R.kRed); hmE.Draw("HIST")

c.cd(3); R.gPad.SetGrid(); hdR.SetLineWidth(2); hdR.Draw("HIST")
c.cd(4); R.gPad.SetGrid(); hmR.SetLineWidth(2); hmR.SetLineColor(R.kRed); hmR.Draw("HIST")

c.Draw()
# c.SaveAs("plots/fee_like_energy_eop_data_vs_mc.png")